# Evaluates the stored responses

- This file evaluates the generated responses using the selected metrics (Pass@1 for MBPP and HumanEval. Exact match for GSM8K).

In [1]:
from irat.utils.settings import env

import IPython
import pandas as pd

import json
import os


# if in the root directory
LLM_name = env('LLM_NAME')
if LLM_name is None:
	raise ValueError('LLM_NAME environment variable is not set.')
print(f'LLM: {LLM_name}')

def switch_to_dir(eval_name):
	# Switch to project's home directory
	while not os.path.exists('README.md'):
		os.chdir('..')
	os.chdir('evaluation')
	eval_dir = f'{eval_name}-responses-{LLM_name.replace(":", "-").replace("/", "-")}'
	print(f'Using', eval_dir)
	os.chdir(eval_dir)

columns_to_test = ['old_rat_answer', 'final_answer']

LLM: Llama-3.3-70B


## Coding evaluations

In [2]:
import re

def get_code(response):
	if not response:
		return None
	# The function was defined in the first code block.
	# Example: ```python  <code>  ```.
	code_blocks = re.findall(r'```python(.*?)```', response, re.DOTALL)
	if code_blocks:
		return code_blocks[0].strip()

	# If code is defined directly without blocks, 
	# run the code to ensure there is no text after the function definition.
	if 'def ' in response:
		try:
			exec(response)
			return response
		except:
			pass
	return None

### HumanEval

Using the official code for HumanEval: https://github.com/openai/human-eval

In [3]:
from human_eval.data import write_jsonl
from human_eval.evaluation import evaluate_functional_correctness
import sys

eval_name = 'human_eval'
switch_to_dir(eval_name)

# Load all responses
responses = []
for file_number in range(1, 164+1):
	filename = f'test_{file_number}.json'
	try:
		with open(filename) as file:
			data = json.load(file)
			responses.append(data['responses'][0])
			responses[-1].update({ 'task_id': data['task_id'] })
	except:
		print(f'Finished extracting {len(responses)} responses from {eval_name} dataset.')
		break

if not responses:
	raise Exception('No responses found. Please check the dataset directory.')

print(f'Loaded {len(responses)} responses from {eval_name} dataset.')

humaneval_results = []

for col in columns_to_test:
	if col not in responses[0]:
		print(f'Column {col} not found in responses data. Stopping.')
		break
	# print(f'Processing column: {col}')
	eval_responses = [
		{ 'task_id': task['task_id'],
			'completion': get_code(task[col]) }
		for task in responses if task.get(col)
	]

	if eval_responses:
		# get the response if it exists for the column
		sample_file = f'humaneval_samples_{col}.jsonl'
		write_jsonl(sample_file, eval_responses)

		# To hide unnecessary warnings or logs.
		old_stderr = sys.stderr
		sys.stderr = open(os.devnull, 'w')
		old_stdout = sys.stdout
		sys.stdout = open(os.devnull, 'w')
		results = evaluate_functional_correctness(sample_file, ignore_incomplete=True)
		sys.stderr = old_stderr
		sys.stdout = old_stdout

		# print(f'{col} results:', results, '\n')
		humaneval_results.append({
			'Column': col,
			'Tested tasks': len(eval_responses),
			'Pass@1 result': f'{results["pass@1"]:.2%}',
		})
		try:
			os.remove(sample_file)
			os.remove(f'{sample_file}_results.jsonl')
		except:
			pass

# clear jupyter output
IPython.display.clear_output()

print('HumanEval results:')
humaneval_df = pd.DataFrame(humaneval_results)

humaneval_first_score = humaneval_df['Pass@1 result'].str.rstrip('%').astype(float).iloc[0]
humaneval_last_score = humaneval_df['Pass@1 result'].str.rstrip('%').astype(float).iloc[-1]
print(f'HumanEval score difference: {humaneval_last_score - humaneval_first_score:.2f}%')

humaneval_df

HumanEval results:
HumanEval score difference: 15.86%


,Column,Tested tasks,Pass@1 result
0,old_rat_answer,164,63.41%
1,final_answer,164,79.27%


### MBPP

In [4]:
eval_name = 'mbpp'
switch_to_dir(eval_name)

# Load all responses
responses = []
for file_number in range(1, 165+1):
	filename = f'test_{file_number}.json'
	try:
		with open(filename) as file:
			data = json.load(file)
			responses.append(data['responses'][0])
			responses[-1].update({
				'task_id': data['task_id'],
				'test_list': data['test_list'],
				'test_imports': data['test_imports'],
				'mbpp_code': data['mbpp_code'],
			})
	except:
		print(f'Extracted {len(responses)} responses.')
		break

def test_code(code: str, tests: list[str], imports: list[str], silent=True) -> bool:
	if not code or not code.strip():
		if not silent:
			print('Warning: Empty code found, skipping tests.')
		return False
	code = code.strip()

	namespace = {}  # to store the imported modules and defined variables
	try:  # Import required modules
		exec('\n'.join(imports), namespace)
	except Exception as e:
		print(f'Import failed with error:', e)
	try:
		exec(code, namespace)  # Execute the code to define the function
		for test_case in tests:
			test_case = test_case.strip()
			if test_case:
				try:
					exec(test_case, namespace)
				except Exception as e:
					return False
	except Exception as e:
		return False

	return True  # All tests passed successfully


# Testing the code using the test cases
mbpp_results = []
for col in columns_to_test:
	correct_count = 0
	total_count = 0
	for sample in responses:
		if col not in sample:
			print(f'Column {col} not found in sample {sample["task_id"]}. Stopping...')
			break
		total_count += 1

		code = get_code(sample[col]) \
					if col != 'mbpp_code' else sample[col]
		if test_code(code, sample['test_list'], sample['test_imports'], silent=True):
			correct_count += 1
		elif col == 'mbpp_code':
			print(f'Task {sample["task_id"]} failed with original MBPP code.')

	if total_count:
		# print(f'{col} results:', accuracy, '\n')
		mbpp_results.append({
			'Column': col,
			'Tested tasks': total_count,
			'Pass@1 result': f'{correct_count / total_count * 100:.2f}%',
		})

IPython.display.clear_output()
print('MBPP results:')
mbpp_df = pd.DataFrame(mbpp_results)
mbpp_first_score = mbpp_df['Pass@1 result'].str.rstrip('%').astype(float).iloc[0]
mbpp_last_score = mbpp_df['Pass@1 result'].str.rstrip('%').astype(float).iloc[-1]
print(f'MBPP score difference: {mbpp_last_score - mbpp_first_score:.2f}%')

mbpp_df
# Original MBPP code is added in mbpp_code column to ensure we executed 
# 	the test cases correctly as per how the dataset was designed.

MBPP results:
MBPP score difference: 12.72%


,Column,Tested tasks,Pass@1 result
0,old_rat_answer,165,63.64%
1,final_answer,165,76.36%


## Mathematical reasoning evaluation

(GSM8K evaluation)

In [5]:
eval_name = 'gsm8k'
switch_to_dir(eval_name)

responses = []

for file_number in range(1, 1319+1):
	filename = f'test_{file_number}.json'
	try:
		with open(filename) as file:
			data = json.load(file)
		responses.append(data['responses'][0])
		responses[-1].update({ 'task_id': data['task_id'], 
								'correct_answer': data['correct_answer'] })
	except:
		print(f'Loaded {len(responses)} responses from {eval_name} dataset.')
		break

print('Total rows:', len(responses))

gsm8k_results = []

def extract_answer(answer):
	# In GSM8K, answer exists after the last #### in the last part.
	# The same function is used for both old-RAT and iRAT.
	answer = answer.split('####')[-1].split('\n')[0].strip()

	# Clean the answer to get only numbers.
	answer = answer.lstrip('$').lstrip('€').strip()  # Remove currency symbols
	answer = answer.rstrip('%').rstrip('.00').strip()  # Remove percentage and trailing zeros
	answer = answer.replace(',', '')  # Remove commas. Example: $1,000.00 -> 1000
	# Select first word by excluding units such as 'years', 'feet', etc. Example: '1000 feet' -> '1000'
	answer = answer.split(' ')[0]
	return answer.strip()

for col in columns_to_test:
	print(f'Processing column: {col}')
	correct_count = 0
	total_count = 0
	for index, row in enumerate(responses, start=1):
		if col not in row:
			continue
		total_count += 1

		# Remove extra characters to measure the exact match.
		correct_answer = extract_answer(row['correct_answer'])
		model_answer = extract_answer(row[col])

		if correct_answer == model_answer:
			correct_count += 1

	if total_count:
		gsm8k_results.append({
			'Column': col,
			'Tested tasks': total_count,
			'Exact Match result': f'{correct_count / total_count * 100:.2f}%',
		})

print('GSM8K results:')
gsm8k_df = pd.DataFrame(gsm8k_results)
gsm8k_first_score = gsm8k_df['Exact Match result'].str.rstrip('%').astype(float).iloc[0]
gsm8k_last_score = gsm8k_df['Exact Match result'].str.rstrip('%').astype(float).iloc[-1]
print(f'GSM8K score difference: {gsm8k_last_score - gsm8k_first_score:.2f}%')

gsm8k_df

Using gsm8k-responses-Llama-3.3-70B
Total rows: 1319
Processing column: old_rat_answer
Processing column: final_answer
GSM8K results:
GSM8K score difference: 8.04%


,Column,Tested tasks,Exact Match result
0,old_rat_answer,1319,81.35%
1,final_answer,1319,89.39%
